<a href="https://colab.research.google.com/github/tushxrgit/CAR-INSURANCE-ASSISTANT/blob/main/CARINS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu
!pip install sentence-transformers
!pip install transformers
!pip install gradio

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline
import gradio as gr

In [ ]:
documents = [

"Car insurance is a financial protection policy that covers damages or losses related to a vehicle. It protects vehicle owners from financial losses caused by accidents, theft, fire, or natural disasters.",

"Third party car insurance is the most basic type of motor insurance. It covers damages caused by the insured vehicle to another person, vehicle, or property.",

"Comprehensive car insurance provides wider protection. It covers third party liabilities as well as damages to the insured vehicle caused by accidents, theft, vandalism, or natural disasters.",

"After a car accident, the driver should inform the insurance company immediately, take photographs of the damage, collect details of the other vehicle, and submit the required documents for the insurance claim.",

"Documents required for car insurance claim usually include a driving license, vehicle registration certificate (RC), insurance policy document, accident photographs, and sometimes a police report.",

"Luxury cars such as BMW, Mercedes-Benz, and Audi usually have higher insurance premiums because their spare parts and repair costs are significantly higher than normal cars.",

"Car insurance policies must be renewed before the expiry date. Driving without valid insurance is illegal in many countries and may result in fines or legal penalties."

]

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
embeddings = model.encode(documents)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

In [ ]:
qa_model = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def retrieve_docs(question):

    q_embedding = model.encode([question])

    distances, indices = index.search(np.array(q_embedding), k=3)

    results = [documents[i] for i in indices[0]]

    return " ".join(results)

In [ ]:
def car_insurance_assistant(question):

    context = retrieve_docs(question)

    result = qa_model(
        question=question,
        context=context
    )

    answer = result["answer"]

    explanation = f"""



{context}
"""

    return explanation

In [ ]:
print(car_insurance_assistant("What is third party car insurance?"))


Answer: the most basic type of motor insurance

Explanation:
Third party car insurance is the most basic type of motor insurance. It covers damages caused by the insured vehicle to another person, vehicle, or property. Car insurance is a financial protection policy that covers damages or losses related to a vehicle. It protects vehicle owners from financial losses caused by accidents, theft, fire, or natural disasters. Comprehensive car insurance provides wider protection. It covers third party liabilities as well as damages to the insured vehicle caused by accidents, theft, vandalism, or natural disasters.



In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🚗 AI Car Insurance Assistant")
    gr.Markdown("Ask questions about car insurance policies, claims, and coverage.")

    chatbot = gr.Chatbot()

    msg = gr.Textbox(
        placeholder="Ask something like: What is third party car insurance?"
    )

    clear = gr.Button("Clear Chat")

    def respond(message, chat_history):

        answer = car_insurance_assistant(message)

        chat_history.append((message, answer))

        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

    clear.click(lambda: None, None, chatbot, queue=False)

demo.launch()

/tmp/ipykernel_6604/2817417650.py:8: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_6604/2817417650.py:8: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2eafb91c32c9d2fa2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
